# Interactive RGB Image to Vegetation Matrix Converter

This notebook processes RGB aerial images to extract vegetation quality and water detection for beaver simulation. Specifically, we consider having a set of images of the same site taken in different moments. The goal of this analysys is to extract a general description of the site in order to obtain a baseline representation to be used as initial condition for future beavers' simulations. Such baseline will be exploited to detect beaver activity over the simulation time and compare it with the real imagery at the end of the site's observation time window.

## Workflow:
0. **Baseline Image** — aggregate all folder images (mean or median) into a single lighting-stable reference
1. **Shadow Correction** — apply CLAHE/histogram shadow correction to the baseline
2. **Vegetation Indices** — compute NDVI, VARI, ExG, CWI on the baseline
3. **Visualize Indices** — inspect index maps, and analyze their values and distribution
4. **Water Detection** — threshold the chosen index on the baseline to produce a water mask
5. **Post-Process Mask** — morphological opening/closing and median filtering to clean the mask
6. **Selected Image** — load the single chosen image, shadow-correct it, project the baseline water mask onto it
7. **Quality Matrix** — score each land pixel relative to the baseline; water pixels = −1


In [ ]:
import sys
import os
from pathlib import Path
import glob
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

cwd = Path.cwd().resolve()
for parent in [cwd, *cwd.parents]:
    if (parent / 'beaversim').is_dir():
        project_root = parent
        break
else:
    raise RuntimeError("Could not locate project root containing 'beaversim'.")

project_root_str = str(project_root)
if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

try:
    from data_acquisition.modules.rgb_to_matrix import (
        load_rgb_image,
        calculate_ndvi,
        calculate_excess_green,
        calculate_cwi,
        detect_water_simple,
        apply_shadow_correction,
        postprocess_water_mask,
    )    
except Exception as e:    
    raise


# Global variables
MAX_COLS = 3
VMIN = 0
VMAX = 1.5
MEAN = (VMAX + VMIN) / 2
print('Modules loaded successfully')

## Configuration: Select Your Example Region

Choose between pre-configured example regions from the Rumney Marsh dataset or set up your own custom region.

In [ ]:

# ============================================================================
# SELECT YOUR IMAGE: 'A2_june', 'A2_july', or 'custom'
# ============================================================================

GENERAL_FOLDER = '../../data_acquisition/example_datasets/RUMNEY_RANCH_RGB/A2/'
REGION = 'A2_july'

# GENERAL_FOLDER = '../../data_acquisition/example_datasets/RUMNEY_RANCH_RGB/B3/'
# REGION = 'B3_july'

# Pre-configured settings for example images
IMAGE_CONFIGS = {
    'A2_june': {
        'name': 'Rumney Marsh Site A2 - June 2018',
        'image_path': GENERAL_FOLDER + 'A2_07262018.PNG',                
        'shadow_correction': True,     
        'remove_mean': True,   
    },
    'A2_july': {
        'name': 'Rumney Marsh Site A2 - July 2018',
        'image_path': GENERAL_FOLDER + 'A2_07172018.PNG',                
        'shadow_correction': True,     
        'remove_mean': True,   
    },
    'A2_june_annotated': {
        'name': 'Rumney Marsh Site A2 - June 2018 - annotated trails',
        'image_path': GENERAL_FOLDER + '/annotated/A2_07262018_annotated.PNG',                
        'shadow_correction': True,     
        'remove_mean': True,   
    },
    'A2_july_annotated': {
        'name': 'Rumney Marsh Site A2 - July 2018 - annotated trails',
        'image_path': GENERAL_FOLDER + '/annotated/A2_07172018_annotated.PNG',                
        'shadow_correction': True,     
        'remove_mean': True,   
    },
     'custom': {
        'name': 'Custom Image',
        'image_path': '',  # UPDATE THIS
        'shadow_correction': True,  # UPDATE THIS
        'remove_mean': True,  # UPDATE THIS
    }
}

# Load selected configuration
config = IMAGE_CONFIGS[REGION]

print(f"✓ {config['name']}")

## Step 0: Compute Baseline Image for Water Detection

Since water is static across seasons, combining all images in `GENERAL_FOLDER` suppresses vegetation variation and makes water features stand out more consistently. In this step we will combine all the images in `GENERAL_FOLDER` and use the outcome to obtain a water mask which will be shareb by all the images in the batch. Such resulting baseline will be used **only for water detection**.

Two aggregation methods are available — select with `BASELINE_METHOD`:
- **Mean**: Simple pixel-wise baseline. Fast and smooth, but biased by outliers (e.g. a beaver trail present in half the images will "ghost in" at 50% intensity).
- **Median**: Pixel-wise temporal median. Ignores minority states; if a structural change (trail, flowers) appears in fewer than half the images it is completely excluded from the baseline. Recommended when structural changes are expected.


In [ ]:
# ============================================================================
# AGGREGATION METHOD
# 'mean'   – pixel-wise average (fast, smooth, sensitive to outliers)
# 'median' – pixel-wise temporal median (robust to structural changes)
# ============================================================================
BASELINE_METHOD = 'median'

# ============================================================================
# Load all images from GENERAL_FOLDER
# ============================================================================
SUPPORTED_EXTS = ('.png', '.jpg', '.jpeg', '.tif', '.tiff')
image_paths = sorted([
    p for p in glob.glob(os.path.join(GENERAL_FOLDER, '*'))
    if p.lower().endswith(SUPPORTED_EXTS)
])

if not image_paths:
    raise FileNotFoundError(f"No images found in GENERAL_FOLDER: {GENERAL_FOLDER}")

print(f"Found {len(image_paths)} image(s) in: {GENERAL_FOLDER}")

# define the target size for resizing images (if needed)
min_height, min_width = None, None
for path in image_paths:
    with Image.open(path) as img:
        w, h = img.size
        if min_height is None or h < min_height:
            min_height = h
        if min_width is None or w < min_width:
            min_width = w

print(f"Smallest image size: height={min_height}, width={min_width}")
TARGET_SIZE = (min_width, min_height)

# Load all images, resize to the same size as the first one
ref_rgb, _ = load_rgb_image(image_paths[0], target_size=TARGET_SIZE)
H, W = ref_rgb.shape[:2]

# Stack all images into a 4D array (N, H, W, 3)
rgb_stack = np.zeros((len(image_paths), H, W, 3), dtype=np.float32)
for i, path in enumerate(image_paths):
    img, _ = load_rgb_image(path, target_size=TARGET_SIZE)
    rgb_stack[i] = img.astype(np.float32)

# Compute both so we can compare them
rgb_mean   = np.clip(rgb_stack.mean(axis=0), 0, 255).astype(np.uint8)
rgb_median = np.clip(np.median(rgb_stack, axis=0), 0, 255).astype(np.uint8)

# Select baseline according to chosen method
if BASELINE_METHOD == 'median':
    rgb_baseline = rgb_median
else:
    rgb_baseline = rgb_mean

print(f"\n✓ Baseline method: {BASELINE_METHOD.upper()}")

# ============================================================================
# Visualize: all individual images
# ============================================================================
n_imgs = len(image_paths)
n_cols = min(MAX_COLS, n_imgs)
n_rows = (n_imgs + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(7 * n_cols, 6 * n_rows))
if n_rows == 1 and n_cols == 1:
    axes = [[axes]]
elif n_rows == 1:
    axes = [axes]
elif n_cols == 1:
    axes = [[ax] for ax in axes]

for i, path in enumerate(image_paths):
    r, c = divmod(i, n_cols)
    axes[r][c].imshow(rgb_stack[i].astype(np.uint8))
    axes[r][c].set_title(os.path.basename(path), fontsize=11, fontweight='bold')
    axes[r][c].axis('off')

for j in range(n_imgs, n_rows * n_cols):
    r, c = divmod(j, n_cols)
    axes[r][c].axis('off')

plt.suptitle(f'All Images in GENERAL_FOLDER\n({GENERAL_FOLDER})',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# ============================================================================
# Compare Mean vs Median side-by-side
# ============================================================================
fig, axes = plt.subplots(1, 3, figsize=(21, 7))

axes[0].imshow(rgb_mean)
axes[0].set_title('Mean\n(sensitive to outliers)', fontsize=13, fontweight='bold',
                  color='navy' if BASELINE_METHOD == 'mean' else 'black')
axes[0].axis('off')

axes[1].imshow(rgb_median)
axes[1].set_title('Median\n(robust to structural changes)', fontsize=13, fontweight='bold',
                  color='navy' if BASELINE_METHOD == 'median' else 'black')
axes[1].axis('off')

# Absolute per-pixel difference (averaged over channels, normalised for visibility)
diff = np.abs(rgb_mean.astype(np.int16) - rgb_median.astype(np.int16)).mean(axis=2)
diff_norm = (diff / diff.max() * 255).astype(np.uint8) if diff.max() > 0 else diff.astype(np.uint8)
im = axes[2].imshow(diff_norm, cmap='viridis')
axes[2].set_title('|Mean − Median|\n(bright = larger divergence)', fontsize=13, fontweight='bold')
axes[2].axis('off')
plt.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)

selected_label = 'MEDIAN  ←  selected' if BASELINE_METHOD == 'median' else 'MEAN  ←  selected'
plt.suptitle(f'Mean vs Median Baseline Comparison  |  {selected_label}',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"✓ rgb_baseline set ({BASELINE_METHOD.upper()}, shape {rgb_baseline.shape})")


## Step 1: Shadow Correction

Apply equalization to the **baseline image** (`rgb_baseline`) computed in Step 0. This produces `rgb_baseline_corrected`, which is used for all subsequent index calculations and water detection.

**Why is this important?**

Shadow correction (using histogram equalization or CLAHE) reduces the impact of varying lighting conditions, such as shadows or changes in weather, on the image. By normalizing brightness and contrast across the baseline image, this step makes the data more invariant to weather and illumination changes. This ensures that subsequent index calculations and water detection are more robust and reliable, as they are less affected by shadows or uneven lighting.

**Best practice:**

It is recommended to first compute the baseline image and then apply shadow correction. This approach ensures that the baseline represents the average or typical appearance of the scene, and the correction is applied consistently. Applying equalization to each image before computing the baseline can introduce inconsistencies, as each image may be normalized differently.

**Methods:**
- `histogram`: Simple histogram equalization per channel
- `clahe`: Adaptive histogram equalization (recommended for aerial images)


In [ ]:
# Apply shadow correction to the baseline image
if config['shadow_correction']:
    rgb_baseline_corrected = apply_shadow_correction(rgb_baseline, method='clahe')
else:
    rgb_baseline_corrected = rgb_baseline.copy()

# Compare original baseline vs shadow-corrected baseline
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

axes[0].imshow(rgb_baseline)
axes[0].set_title('Baseline Image (Before Correction)', fontsize=14, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(rgb_baseline_corrected)
axes[1].set_title('Baseline Image (Shadow Corrected)', fontsize=14, fontweight='bold')
axes[1].axis('off')

plt.tight_layout()
plt.show()

print("✓ rgb_baseline_corrected ready")


## Step 2: Calculate Vegetation Indices

Calculate four indices from the shadow-corrected **baseline image** (`rgb_baseline_corrected`), three vegetation indices and one water index, all rescaled between [-1,1]:

**NDVI (Normalized Difference Vegetation Index)**
- Formula: `(Green - Red) / (Green + Red)` (visible approximation)
- High values indicate dense vegetation

**ExG (Excess Green Index)**
- Formula: `2*Green - Red - Blue`
- High values emphasize green vegetation, useful for separating plants from soil/water.

**CWI (Color Water Index)**
- Formula: `log(Blue / Red)`
- High values indicate water; negative values indicate non-water (vegetation, soil).


In [ ]:
# Compute vegetation indices on the shadow-corrected baseline
ndvi_baseline = calculate_ndvi(rgb_baseline_corrected, method='visible', remove_mean=config['remove_mean'])
exg_baseline  = calculate_excess_green(rgb_baseline_corrected, remove_mean=config['remove_mean'])
cwi_baseline  = calculate_cwi(rgb_baseline_corrected, remove_mean=config['remove_mean'])


## Step 3: Visualize Baseline Indices

Inspect all four indices derived from the baseline to understand vegetation patterns and locate water bodies.


In [ ]:
# Visualize all indices
fig, axes = plt.subplots(1, 3, figsize=(20, 13))

vmin_index = 0.0
vmax_index = 1.0

# NDVI
im1 = axes[0].imshow(ndvi_baseline, cmap='RdYlGn', vmin=vmin_index, vmax=vmax_index)
axes[0].set_title('NDVI (Vegetation Index)', fontsize=14, fontweight='bold')
axes[0].axis('off')
cbar1 = plt.colorbar(im1, ax=axes[0], fraction=0.046, pad=0.04, shrink=0.25)
cbar1.set_label('NDVI Value', fontsize=20)
cbar1.ax.tick_params(labelsize=16)

# ExG
im3 = axes[1].imshow(exg_baseline, cmap='RdYlGn', vmin=vmin_index, vmax=vmax_index)
axes[1].set_title('ExG (Excess Green)', fontsize=14, fontweight='bold')
axes[1].axis('off')
cbar3 = plt.colorbar(im3, ax=axes[1], fraction=0.046, pad=0.04, shrink=0.25)
cbar3.set_label('ExG Value', fontsize=20)
cbar3.ax.tick_params(labelsize=16)

# CWI (use blue-to-brown colormap for water)
im4 = axes[2].imshow(cwi_baseline, cmap='RdYlBu', vmin=vmin_index, vmax=vmax_index)
axes[2].set_title('CWI (Color Water Index)', fontsize=14, fontweight='bold')
axes[2].axis('off')
cbar4 = plt.colorbar(im4, ax=axes[2], fraction=0.046, pad=0.04, shrink=0.25)
cbar4.set_label('CWI Value (High=Water)', fontsize=20)
cbar4.ax.tick_params(labelsize=16)

plt.tight_layout()
plt.show()

## Step 4: Index Distributions

Compare NDVI, VARI, ExG, and CWI histograms in order to define a water threshold. Run this step to guide the manual threshold choice to be used for the water detection. Specifically:

- **NDVI / VARI / ExG**: water pixels fall *below* the threshold
- **CWI**: water pixels fall *above* the threshold


In [ ]:
# Extract flattened index data for visualization
ndvi_flat = ndvi_baseline.flatten()
exg_flat  = exg_baseline.flatten()
cwi_flat  = cwi_baseline.flatten()

# Create 2x2 grid of histograms showing index distributions and automatic thresholds
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# NDVI histogram
axes[0].hist(ndvi_flat, bins=100, color='green', alpha=0.6, label='NDVI')
axes[0].axvline(0, color='black', linestyle='-', linewidth=1, alpha=0.3)
axes[0].set_xlabel('NDVI Value')
axes[0].set_ylabel('Frequency')
axes[0].set_title('NDVI Distribution (Water < threshold)', fontweight='bold', fontsize=12)
axes[0].legend()
axes[0].grid(alpha=0.3)

# ExG histogram
axes[1].hist(exg_flat, bins=100, color='purple', alpha=0.6, label='ExG')
axes[1].axvline(0, color='black', linestyle='-', linewidth=1, alpha=0.3)
axes[1].set_xlabel('ExG Value')
axes[1].set_ylabel('Frequency')
axes[1].set_title('ExG Distribution (Water < threshold)', fontweight='bold', fontsize=12)
axes[1].legend()
axes[1].grid(alpha=0.3)

# CWI histogram
axes[2].hist(cwi_flat, bins=100, color='cyan', alpha=0.6, label='CWI')
axes[2].axvline(0, color='black', linestyle='-', linewidth=1, alpha=0.3)
axes[2].set_xlabel('CWI Value')
axes[2].set_ylabel('Frequency')
axes[2].set_title('CWI Distribution (Water > threshold)', fontweight='bold', fontsize=12)
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.suptitle('Index Distributions with Automatic Thresholds (Baseline Image)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()



## Step 5: Water Detection

Detect water using the **baseline image** (`rgb_baseline_corrected`). The baseline suppresses seasonal vegetation variation, making water features stable and easier to threshold. 

Indices available: `ndvi_baseline`, `vari_baseline`, `exg_baseline`, `cwi_baseline`.


In [ ]:
# Choose which index to use for water detection
# Options: 'NDVI', 'VARI', 'ExG', or 'CWI'
vegetation_index_choice = 'ExG'  # Change this to experiment

# Select the baseline index
if vegetation_index_choice == 'NDVI':
    veg_index = ndvi_baseline
elif vegetation_index_choice == 'ExG':
    veg_index = exg_baseline
elif vegetation_index_choice == 'CWI':
    veg_index = cwi_baseline
else:
    raise ValueError(f"Invalid vegetation_index_choice: {vegetation_index_choice}. "
                     f"Choose from 'NDVI', 'ExG', or 'CWI'.")

# ============================================================================
# MANUAL THRESHOLD OVERRIDE (Optional)
# ============================================================================
use_manual_threshold = True


index_threshold = 0.33   # For NDVI/ExG: water < threshold
                                    # For CWI: water > threshold

# ============================================================================
# THRESHOLD SELECTION
# ============================================================================

print(f"  Selected index: {vegetation_index_choice} (baseline)")
print(f"  Threshold: {index_threshold:.3f}")

    
water_mask_raw = detect_water_simple(
    rgb_baseline_corrected,
    veg_index,
    index_name=vegetation_index_choice,
    index_threshold=index_threshold
)

# Store raw detection for post-processing
water_mask = water_mask_raw.copy()
print(f"\n✓ Water detection complete (based on baseline built from {len(image_paths)} images)")

# Visualize: baseline image, raw mask, and overlay on baseline image
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(rgb_baseline)
axes[0].set_title('Baseline (water detection input)', fontsize=13, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(water_mask_raw, cmap='Blues')
axes[1].set_title('Raw Water Detection', fontsize=13, fontweight='bold')
axes[1].axis('off')

rgb_overlay = rgb_baseline.copy()
rgb_overlay[water_mask_raw] = [30, 144, 255]
axes[2].imshow(rgb_overlay)
axes[2].set_title('Water Mask Overlay\n(on baseline image)', fontsize=13, fontweight='bold')
axes[2].axis('off')

plt.tight_layout()
plt.show()

water_pixels = np.sum(water_mask_raw)
total_pixels = water_mask_raw.size
print(f"  Water coverage: {100*water_pixels/total_pixels:.2f}% ({water_pixels:,} pixels)")


## Step 6: Post-Process Water Mask

Apply morphological operations to clean up the water detection:
1. **Opening (erosion → dilation)**: Removes small isolated false positives without permanently shrinking legitimate water bodies
2. **Median filter**: Further removes pissible outliers
3. **Closing (dilation → erosion)**: Fills small holes and connects nearby water fragments without adding false positives outside water regions

Adjust the kernel sizes based on your image resolution and water feature sizes. Smaller kernels (3-7) for fine details, larger kernels (9-15) for aggressive cleanup.


In [ ]:
# Verify water_mask_raw exists from previous step
if 'water_mask_raw' not in locals():
    raise RuntimeError("Error: water_mask_raw not found. Please run Step 4 (water detection) first!")

initial_water_pixels = np.sum(water_mask_raw)

# Configuration: Adjust these kernel sizes based on your needs
# - Smaller kernels (3-5): Keep fine details, gentle cleanup
# - Larger kernels (7-15): More aggressive noise removal and connection
water_mask = postprocess_water_mask(
    water_mask_raw,
    opening_sizes=[8],      # List of opening kernels (removes small false positives)
    median_sizes=[],             # List of median filter kernels (optional)
    closing_sizes=[30],      # List of closing kernels (fills holes, connects fragments)
    apply_opening=True,
    apply_median=False,
    apply_closing=True
)

# Compare before and after - Two visualizations
# 1. Side-by-side comparison of masks
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Raw detection
axes[0].imshow(water_mask_raw, cmap='Blues')
axes[0].set_title('Raw Water Detection', fontsize=14, fontweight='bold')
axes[0].axis('off')

# Post-processed
axes[1].imshow(water_mask, cmap='Blues')
axes[1].set_title('After Post-Processing', fontsize=14, fontweight='bold')
axes[1].axis('off')

# Difference (what changed)
difference = water_mask.astype(int) - water_mask_raw.astype(int)
diff_rgb = np.zeros((*water_mask.shape, 3), dtype=np.uint8)
diff_rgb[difference == -1] = [255, 0, 0]    # Removed: red
diff_rgb[difference == 1]  = [0, 255, 0]    # Added: green
diff_rgb[difference == 0]  = [128, 128, 128] # Unchanged: gray
axes[2].imshow(diff_rgb)
axes[2].set_title('Changes (Red=Removed, Green=Added)', fontsize=14, fontweight='bold')
axes[2].axis('off')

plt.tight_layout()
plt.show()

# 2. RGB overlays comparison (on baseline image)
fig, axes = plt.subplots(1, 2, figsize=(20, 7))

# Before post-processing
rgb_overlay_raw = rgb_baseline.copy()
rgb_overlay_raw[water_mask_raw] = [30, 144, 255]
axes[0].imshow(rgb_overlay_raw)
axes[0].set_title('Baseline RGB + Water (Before Post-Processing)', fontsize=14, fontweight='bold')
axes[0].axis('off')

# After post-processing
rgb_overlay_clean = rgb_baseline.copy()
rgb_overlay_clean[water_mask] = [30, 144, 255]
axes[1].imshow(rgb_overlay_clean)
axes[1].set_title('Baseline RGB + Water (After Post-Processing)', fontsize=14, fontweight='bold')
axes[1].axis('off')

plt.tight_layout()
plt.show()

# Statistics
water_pixels_final = np.sum(water_mask)
total_pixels = water_mask.size
print(f"✓ Water mask ready  |  {100*water_pixels_final/total_pixels:.2f}% water  ({water_pixels_final:,} px)")


## Step 7: Load Selected Image & Apply Water Mask

Load the specific image chosen in the config (`REGION`), apply the water mask produced in Step 6, and shadow-correct it. This is the image that will be used for vegetation quality scoring. The baseline so far was only needed for stable water detection.


In [ ]:
# Load the individually selected image
rgb_original, metadata = load_rgb_image(config['image_path'], target_size=TARGET_SIZE)

# Shadow correction
if config['shadow_correction']:
    rgb = apply_shadow_correction(rgb_original, method='clahe')
    pass
else:
    rgb = rgb_original.copy()

# Apply water mask
rgb_masked = rgb.copy()
rgb_masked[water_mask] = [30, 144, 255]   # paint water pixels dodger-blue

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(20, 7))

axes[0].imshow(rgb_original)
axes[0].set_title(f'Original Image\n({config["name"]})', fontsize=13, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(rgb)
axes[1].set_title('Shadow Corrected', fontsize=13, fontweight='bold')
axes[1].axis('off')

axes[2].imshow(rgb_masked)
axes[2].set_title('Shadow Corrected + Water Mask\n(blue = detected water)', fontsize=13, fontweight='bold')
axes[2].axis('off')

plt.suptitle(f'Selected Image with Baseline Water Mask applied  |  {REGION}',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"✓ rgb shape {rgb.shape}  |  water_mask shape {water_mask.shape}")


## Step 8: Histogram Matching (Weather Normalization)

Match the histogram of the **shadow-corrected selected image** (`rgb`) to that of the
**shadow-corrected baseline** (`rgb_baseline_corrected`), using **land pixels only** as
the reference distribution. This corrects for differences in overall brightness and contrast
caused by varying weather and lighting conditions between acquisition dates, so that the
vegetation indices in Step 7 reflect genuine vegetation change rather than illumination artefacts.

The result is `rgb_matched`, which replaces `rgb` as input for index computation in Step 9.


In [ ]:
from skimage.exposure import match_histograms

# Resize baseline to match selected image resolution if needed
base_h, base_w = rgb_baseline_corrected.shape[:2]
sel_h,  sel_w  = rgb.shape[:2]
if (base_h, base_w) != (sel_h, sel_w):
    ref_for_match = np.array(
        Image.fromarray(rgb_baseline_corrected).resize((sel_w, sel_h), Image.BILINEAR)
    ).astype(np.uint8)
else:
    ref_for_match = rgb_baseline_corrected

# Histogram-match selected image to baseline using land-only pixels as reference.
# match_histograms requires both arrays to have the same ndim, so we flatten the
# source to 1D, match against the 1D land reference, then reshape back.
LAND = ~water_mask

rgb_matched = np.empty_like(rgb)
for ch in range(3):
    src_flat = rgb[:, :, ch].astype(np.float32).ravel()
    ref_land = ref_for_match[:, :, ch].astype(np.float32)[LAND]   # 1-D land-only reference
    matched_flat = match_histograms(src_flat, ref_land)
    rgb_matched[:, :, ch] = np.clip(matched_flat.reshape(rgb.shape[:2]), 0, 255).astype(np.uint8)

print(f'✓ rgb_matched  |  shape {rgb_matched.shape}')

# ── Visualize: images before/after matching ───────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(21, 7))

axes[0].imshow(rgb_baseline_corrected)
axes[0].set_title('Baseline (reference)', fontsize=13, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(rgb)
axes[1].set_title('Selected — shadow corrected\n(before matching)', fontsize=13, fontweight='bold')
axes[1].axis('off')

axes[2].imshow(rgb_matched)
axes[2].set_title('Selected — histogram matched\n(weather normalized)', fontsize=13, fontweight='bold')
axes[2].axis('off')

plt.suptitle(f'Histogram Matching  |  {REGION}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# ── Per-channel histogram comparison (land pixels only) ───────────────────────
# Row 1: Baseline vs Selected (before matching) — shows the original difference
# Row 2: Baseline vs Matched (after matching)   — shows the correction applied
fig, hist_axes = plt.subplots(2, 3, figsize=(20, 8))
ch_colors = ['red', 'green', 'blue']
ch_names  = ['Red', 'Green', 'Blue']
for ch in range(3):
    # --- Row 0: before matching ---
    hist_axes[0, ch].hist(ref_for_match[LAND, ch].ravel(), bins=100, alpha=0.55,
                          color='gray',       label='Baseline', density=True)
    hist_axes[0, ch].hist(rgb[LAND, ch].ravel(),            bins=100, alpha=0.55,
                          color=ch_colors[ch], label='Selected (before)', density=True)
    hist_axes[0, ch].set_title(f'{ch_names[ch]} — before matching', fontweight='bold')
    hist_axes[0, ch].legend(fontsize=9)
    hist_axes[0, ch].set_xlabel('Pixel value')
    hist_axes[0, ch].grid(alpha=0.3)

    # --- Row 1: after matching ---
    hist_axes[1, ch].hist(ref_for_match[LAND, ch].ravel(), bins=100, alpha=0.55,
                          color='gray',       label='Baseline', density=True)
    hist_axes[1, ch].hist(rgb_matched[LAND, ch].ravel(),   bins=100, alpha=0.55,
                          color=ch_colors[ch], label='Matched (after)', density=True,
                          linewidth=2)
    hist_axes[1, ch].set_title(f'{ch_names[ch]} — after matching', fontweight='bold')
    hist_axes[1, ch].legend(fontsize=9)
    hist_axes[1, ch].set_xlabel('Pixel value')
    hist_axes[1, ch].grid(alpha=0.3)

plt.suptitle('Per-channel Histograms — Land Pixels Only\n'
             'Row 1: before matching  |  Row 2: after matching',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Compute vegetation indices on the shadow-corrected baseline
ndvi_baseline_land = calculate_ndvi(rgb_baseline_corrected, method='visible', remove_mean=config['remove_mean'], mask=water_mask, mean=MEAN)
exg_baseline_land  = calculate_excess_green(rgb_baseline_corrected, remove_mean=config['remove_mean'], mask=water_mask, mean=MEAN)
cwi_baseline_land  = calculate_cwi(rgb_baseline_corrected, remove_mean=config['remove_mean'], mask=water_mask, mean=MEAN)

## Step 10: Vegetation Quality Score & Final Matrix

**Why use the baseline as a reference for vegetation quality?**

The baseline image is constructed to represent the typical or average state of the site, aggregating information from multiple images across time. By using the baseline both for water detection and as a reference for vegetation scoring, we achieve two main goals:

1. **Stable Water Detection:** Water features are generally static and less affected by seasonal changes. Detecting water on the baseline suppresses transient vegetation effects and ensures a consistent water mask across all images.

2. **Relative Vegetation Assessment:** Comparing the selected image’s vegetation index to the baseline allows us to quantify how much the current state deviates from the site’s typical condition. This is especially useful for monitoring changes over time, such as the impact of beaver activity, drought, or restoration efforts. A score of 0.5 means the vegetation is at its seasonal norm; values below 0.5 indicate depletion, and above 0.5 indicate lusher-than-normal conditions.

This approach is optional: you can also use the raw vegetation index from the selected image as a direct quality score. However, referencing the baseline provides a more robust, context-aware measure of vegetation health, accounting for site-specific variability and long-term trends.

Compute a normalised vegetation quality score from the **selected image**, referenced against the **median baseline**:

$$\text{score} = \frac{1 + \text{INDEX}_{\text{selected}}}{2 \cdot (1 + \text{INDEX}_{\text{baseline}} + \varepsilon)}$$

- Range: `[0, 1]` on land pixels &mdash; `0.5` = at the seasonal norm, `< 0.5` = depleted, `> 0.5` = lush
- Water pixels are set to `−1`
- `ε = 0.05` prevents division blow-up where the baseline index is near −1

Select `SCORE_INDEX` to choose which vegetation index drives the quality score.


In [ ]:
# ============================================================================
# SCORING INDEX SELECTION
# Options: 'NDVI', 'ExG', 'CWI'
# ============================================================================
SCORE_INDEX = 'NDVI'  # Change this to select which index to use for scoring
FINAL_VALUE = 'index_selected'  # Options: 'vegetation_score' or 'index_selected'
EPS = 1.0  # Stability epsilon for near-zero baseline pixels

# --- Compute index on histogram-matched selected image ---
if SCORE_INDEX == 'NDVI':
    index_selected  = calculate_ndvi(rgb_matched, method='visible', remove_mean=config['remove_mean'], mask=water_mask, mean=MEAN)
    index_baseline = ndvi_baseline_land
elif SCORE_INDEX == 'ExG':
    index_selected = calculate_excess_green(rgb_matched, remove_mean=config['remove_mean'], mask=water_mask, mean=MEAN)
    index_baseline = exg_baseline_land
elif SCORE_INDEX == 'CWI':
    index_selected = calculate_cwi(rgb_matched, remove_mean=config['remove_mean'], mask=water_mask, mean=MEAN)
    index_baseline = cwi_baseline_land
else:
    raise ValueError(f"Unknown SCORE_INDEX: '{SCORE_INDEX}'. Choose from 'NDVI', 'ExG', 'CWI'.")

# --- Final matrix: land = score, water = -1 ---
if FINAL_VALUE == 'index_selected':    
    final_matrix = index_selected.copy()    
    baseline_matrix = index_baseline.copy()
else:
    
    # --- Score: (1 + index_selected) / (2 * (1 + index_baseline + eps)) → [0, 1] ---
    vegetation_score = np.full_like(index_selected, fill_value=np.nan, dtype=np.float32)
    vegetation_score[LAND] = (1.0 + index_selected[LAND] + EPS) / (2.0 * (1.0 + index_baseline[LAND] + EPS))    

    # --- Baseline image: INDEX on land, -1 on water ---
    baseline_score = np.full_like(index_baseline, fill_value=np.nan, dtype=np.float32)
    baseline_score[LAND] = (1.0 + index_baseline[LAND] + EPS) / (2.0 * (1.0 + index_baseline[LAND] + EPS))
    baseline_matrix = baseline_score.copy()    
    
    final_matrix = vegetation_score.copy()
    baseline_matrix = baseline_score.copy()
    
    # Water pixels → -1
    print(f"✓ Vegetation score computed using {SCORE_INDEX}")
    print(f"  Land score range : [{vegetation_score[LAND].min():.3f}, {vegetation_score[LAND].max():.3f}]")
    print(f"  Mean land score  : {vegetation_score[LAND].mean():.3f}")
    print(f"  Water pixels     : {water_mask.sum():,} set to −1")
    

# --- Visualize: 2×2 grid ---
# Row 0 : raw index maps (comparable, same vmin/vmax)
# Row 1 : normalised score matrices
cmap_idx   = plt.cm.RdYlGn.copy()
cmap_idx.set_bad('dodgerblue')    # water in blue
cmap_score = plt.cm.RdYlGn.copy()
cmap_score.set_bad('dodgerblue')

# Shared index colour limits: use the combined land range
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# (0,0) Baseline raw index
im00 = axes[0].imshow(baseline_matrix, cmap=cmap_idx, vmin=VMIN, vmax=VMAX)
axes[0].set_title(f'Baseline {SCORE_INDEX} (raw index)', fontsize=13, fontweight='bold')
axes[0].axis('off')
cbar = plt.colorbar(im00, ax=axes[0], fraction=0.046, pad=0.04, shrink=0.7)
cbar.ax.tick_params(labelsize=16)
cbar.ax.set_ylabel(f'{SCORE_INDEX}', fontsize=20)

# (0,1) Final quality matrix
im01 = axes[1].imshow(final_matrix, cmap=cmap_score, vmin=VMIN, vmax=VMAX)
axes[1].set_title(f'Final Quality Matrix\n(land: [0,1] | water: −1)', fontsize=13, fontweight='bold')
axes[1].axis('off')
cbar = plt.colorbar(im01, ax=axes[1], fraction=0.046, pad=0.04, shrink=0.7)
cbar.ax.tick_params(labelsize=16)
cbar.ax.set_ylabel(f'{SCORE_INDEX}', fontsize=20)

plt.suptitle(f'Vegetation Quality Matrix  |  {REGION}  |  Index: {SCORE_INDEX}\n'
             'Top: raw index comparison  |  Bottom: normalised scores',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
